<a href="https://colab.research.google.com/github/eman-25/Bahrain-Customer-Churn-Prediction/blob/main/Bahrain_Customer_Churn_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

print("1️⃣ Loading and cleaning data...")
# ==========================================
# Step 1: Load and Prepare Data
# ==========================================
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# Drop unnecessary identifier column
df.drop(columns=['customerID'], inplace=True)

# Handle missing values and convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Encode target column (Churn) to binary integers (1 or 0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Convert text and categorical features into numerical variables (One-Hot Encoding)
df = pd.get_dummies(df, drop_first=True)


print("2️⃣ Splitting data and training the Balanced Ensemble Model...")
# ==========================================
# Step 2: Split Data and Train Balanced Models (Real Data Only)
# ==========================================
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define models with class_weight='balanced' to handle imbalance authentically
lr_model = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
dt_model = DecisionTreeClassifier(class_weight='balanced', random_state=42)
rf_model = RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42)
gb_model = GradientBoostingClassifier(random_state=42)

# Combine models using a Voting Classifier
balanced_ensemble = VotingClassifier(
    estimators=[
        ('lr', lr_model),
        ('dt', dt_model),
        ('rf', rf_model),
        ('gb', gb_model)
    ],
    voting='soft'
)

# Train the balanced ensemble model
balanced_ensemble.fit(X_train, y_train)

# Calculate accuracy
preds = balanced_ensemble.predict(X_test)
acc = accuracy_score(y_test, preds)
print(f"🔥 Final Balanced Ensemble Model Accuracy: {acc * 100:.2f}%\n")


print("3️⃣ Extracting Macro Strategic Portfolio Insights...")
# ==========================================
# Step 3: Extract Strategic Report & Recommendations
# ==========================================
churn_probabilities = balanced_ensemble.predict_proba(X_test)[:, 1]
high_risk_customers = churn_probabilities > 0.5
overall_risk_percentage = (high_risk_customers.sum() / len(churn_probabilities)) * 100

print("=" * 55)
print("📊 Comprehensive Strategic Portfolio Report")
print("=" * 55)
print(f"• Overall General Churn Risk Percentage: {overall_risk_percentage:.1f}%")
print(f"• Estimated Customers at Risk in Test Sample: {high_risk_customers.sum()} customers")
print("\n• Proposed Strategic Recommendation:")
if overall_risk_percentage > 15:
    print("  -> High risk level detected! Suggest launching general discount campaigns or incentive contests to boost engagement.")
else:
    print("  -> Situation is stable. Focus individually on high-risk customers.")
print("=" * 55)


print("\n4️⃣ Saving the model for web interface deployment...")
# ==========================================
# Step 4: Save Model and Feature Columns
# ==========================================
joblib.dump(balanced_ensemble, 'churn_model.pkl')
joblib.dump(X.columns.tolist(), 'model_columns.pkl')

print("✅ Files (churn_model.pkl and model_columns.pkl) saved successfully and ready for the next phase!")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set up clean and professional plot styling
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. First plot: Customer Churn Distribution (Stayed vs. Churned) with modern Seaborn syntax
sns.countplot(x=df['Churn'], hue=df['Churn'], ax=axes[0], palette=['#2ecc71', '#e74c3c'], legend=False)
axes[0].set_title('Customer Distribution: Stayed (0) vs. Churned (1)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Customer Status (0 = Stayed, 1 = Churned)')
axes[0].set_ylabel('Number of Customers')

# 2. Second plot: Top 5 features driving churn (extracted from Random Forest within the Ensemble)
rf_sub_model = ensemble_model.named_estimators_['rf']
feature_importances = pd.Series(rf_sub_model.feature_importances_, index=X.columns)
top_features = feature_importances.nlargest(5)

top_features.plot(kind='barh', ax=axes[1], color='#3498db')
axes[1].set_title('Top 5 Features Driving Customer Churn', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Importance Score')
axes[1].set_ylabel('Features')

# Adjust layout for a clean, readable presentation
plt.tight_layout()
plt.show()

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib

# Page configuration
st.set_page_config(
    page_title="B2B Churn Prediction Dashboard",
    page_icon="📊",
    layout="wide"
)

# Load the saved model and columns
@st.cache_resource
def load_model_and_data():
    model = joblib.load('churn_model.pkl')
    model_columns = joblib.load('model_columns.pkl')
    return model, model_columns

model, model_columns = load_model_and_data()

# App Title
st.title("📊 B2B Customer Churn Prediction & Decision Support System")
st.markdown("A professional AI-powered tool to predict customer churn, identify risk drivers, and drive strategic retention in Bahrain.")

# Sidebar Navigation
st.sidebar.header("Navigation Menu")
app_mode = st.sidebar.selectbox("Choose Dashboard View", ["Macro Portfolio Insights", "Individual Customer Prediction"])

# ==========================================
# VIEW 1: MACRO PORTFOLIO INSIGHTS (Strategic Report)
# ==========================================
if app_mode == "Macro Portfolio Insights":
    st.header("🏢 Macro Strategic Portfolio Dashboard")
    st.markdown("Overview of general customer health and portfolio-level risk assessment.")

    # Mocking macro stats based on test evaluation
    col1, col2, col3 = st.columns(3)
    with col1:
        st.metric(label="Overall Churn Risk", value="20.3%", delta="-1.2% vs last month", delta_color="inverse")
    with col2:
        st.metric(label="Estimated Customers at Risk", value="286", delta="+14", delta_color="inverse")
    with col3:
        st.metric(label="Potential Revenue at Risk", value="14,500 BHD", delta_color="inverse")

    st.markdown("---")
    st.subheader("💡 Executive Strategic Recommendations")
    st.info(
        "**High Risk Level Detected!**\n\n"
        "1. Launch general discount campaigns or interactive retention contests.\n"
        "2. Focus retention teams on high-usage fiber optic subscribers using electronic check payment methods."
    )

# ==========================================
# VIEW 2: INDIVIDUAL CUSTOMER PREDICTION (Micro Insights)
# ==========================================
elif app_mode == "Individual Customer Prediction":
    st.header("👤 Individual Customer Churn Predictor")
    st.markdown("Enter customer details below to predict their churn probability and uncover specific risk drivers.")

    with st.form("prediction_form"):
        col1, col2 = st.columns(2)

        with col1:
            tenure = st.slider("Tenure (Months with company)", 0, 72, 12)
            monthly_charges = st.number_input("Monthly Charges (BHD)", 10.0, 150.0, 70.0)
            total_charges = st.number_input("Total Charges (BHD)", 10.0, 8000.0, 850.0)

        with col2:
            contract_type = st.selectbox("Contract Type", ["Month-to-month", "One year", "Two year"])
            internet_service = st.selectbox("Internet Service", ["DSL", "Fiber optic", "No"])
            payment_method = st.selectbox("Payment Method", ["Electronic check", "Mailed check", "Bank transfer", "Credit card"])

        submit_button = st.form_submit_button(label="Predict Churn Risk")

    if submit_button:
        # Create a blank input dataframe matching model columns
        input_data = pd.DataFrame(0, index=[0], columns=model_columns)

        # Populate numerical values
        if 'tenure' in input_data.columns:
            input_data['tenure'] = tenure
        if 'MonthlyCharges' in input_data.columns:
            input_data['MonthlyCharges'] = monthly_charges
        if 'TotalCharges' in input_data.columns:
            input_data['TotalCharges'] = total_charges

        # Populate encoded categorical columns safely
        if contract_type == "One year" and 'Contract_One year' in input_data.columns:
            input_data['Contract_One year'] = 1
        elif contract_type == "Two year" and 'Contract_Two year' in input_data.columns:
            input_data['Contract_Two year'] = 1

        if internet_service == "Fiber optic" and 'InternetService_Fiber optic' in input_data.columns:
            input_data['InternetService_Fiber optic'] = 1
        elif internet_service == "No" and 'InternetService_No internet service' in input_data.columns:
            input_data['InternetService_No internet service'] = 1

        if payment_method == "Electronic check" and 'PaymentMethod_Electronic check' in input_data.columns:
            input_data['PaymentMethod_Electronic check'] = 1
        # (Other categories default to 0 like mailed check, bank transfer, etc.)

        # Make prediction
        prediction_prob = model.predict_proba(input_data)[0][1]
        prediction = model.predict(input_data)[0]

        st.markdown("---")
        st.subheader("🎯 Prediction Results")

        if prediction_prob > 0.5:
            st.error(f"⚠️ **High Risk of Churn!** (Probability: {prediction_prob * 100:.1f}%)")
            st.markdown("**Key Risk Drivers Identified:**")
            st.markdown("- High monthly billing relative to tenure length.")
            st.markdown("- Subscribed to high-risk service tiers without active engagement discounts.")
            st.markdown("💡 **Recommended Action:** Offer an immediate 15% discount on the next billing cycle or assign a dedicated customer success representative.")
        else:
            st.success(f"✅ **Customer is Stable / Low Risk** (Churn Probability: {prediction_prob * 100:.1f}%)")
            st.markdown("💡 **Recommended Action:** Maintain standard engagement and periodic check-ins.")

In [ ]:
!pip install streamlit -q


In [ ]:
!pip install pyngrok -q

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("3LFKS3JQuOj2czS87Cph7W74BF6_4rbGfPMtGPcD6RoPPj3Xd")

public_url = ngrok.connect(8501)
print(public_url)

In [ ]:
!streamlit run app.py &